<a href="https://colab.research.google.com/github/lmendezayl/uba-optimizacion-tp1/blob/main/tp1_optimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import Pkg; Pkg.add("RDatasets")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
   2137.3 ms  ✓ CoreMath_jll
   2054.3 ms  ✓ EpollShim_jll
   2498.9 ms  ✓ LLVMOpenMP_jll
   2086.8 ms  ✓ Imath_jll
   2592.3 ms  ✓ LibTracyClient_jll
   2270.1 ms  ✓ Giflib_jll
   2024.1 ms  ✓ CRlibm_jll
   2369.3 ms  ✓ isoband_jll
   2974.5 ms  ✓ MbedTLS_jll
   2240.5 ms  ✓ libsixel_jll
   2170.7 ms  ✓ OpenBLASConsistentFPCSR_jll
   3988.9 ms  ✓ CodecZstd
   6825.2 ms  ✓ Ghostscript_jll
   6759.6 ms  ✓ Reactant_jll
   4861.8 ms  ✓ Enzyme_jll
  12010.0 ms  ✓ JpegTurbo
  23145.5 ms  ✓ PNGFiles
   6135.0 ms  ✓ SpecialFunctions → SpecialFunctionsChainRulesCoreExt
  15718.2 ms  ✓ WeightInitializers
   1636.9 ms  ✓ StatsFuns → StatsFunsInverseFunctionsExt
  36529.6 ms  ✓ GridLayoutBase
   6957.3 ms  ✓ StatsFuns → StatsFunsChainRulesCoreExt
  20661.3

In [ ]:
using LinearAlgebra, RDatasets, Random, Statistics, Plots

In [ ]:
iris = dataset("datasets", "iris")
X = Matrix(iris[:, 1:4])
y = iris.Species

150-element CategoricalArrays.CategoricalArray{String,1,UInt8}:
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 "setosa"
 ⋮
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"
 "virginica"

In [74]:
# utils_tools.jl
function split(X, y; dims=1, ratio_train=0.8)
    n = length(y)

    n_train = round(Int, ratio_train*n) #Redondeamos el corte
    i_rand = randperm(n)				# Permutamos los indices
    i_train = i_rand[1:n_train] 		# Usamos el 80% para train
    i_test = i_rand[n_train+1:end] 		# El resto lo usamos para test

    X_train = X[i_train,:]
    y_train = y[i_train]
    X_test  = X[i_test, :]
    y_test  = y[i_test]
    return X_train, y_train, X_test, y_test

end

function normalize(X_train, X_test; dims=1)
    col_mean = mean(X_train; dims)
    col_std = std(X_train; dims)

    return (X_train .- col_mean) ./ col_std, (X_test .- col_mean) ./ col_std
end

function onehot(y, classes)
	y_onehot = zeros(length(classes), length(y))
	num_of_class = 1:length(classes)

	for i in 1:length(y)
		y_onehot[:,i] = y[i].==classes
	end
	return y_onehot
end

function prepare_data(X, y; do_normal=true, do_onehot=true, kwargs...)
    X_train, y_train, X_test, y_test = split(X, y)

    if do_normal
        X_train, X_test = normalize(X_train, X_test; kwargs...)
    end

    classes = unique(y)

    if do_onehot
        y_train = onehot(y_train, classes)
        y_test = onehot(y_test, classes)
    end

    return X_train', y_train, X_test', y_test, classes
end


mean_tuple(d::AbstractArray{<:Tuple}) = Tuple([mean([d[k][i] for k in 1:length(d)]) for i in 1:length(d[1])])

grad_total(modelo,grad,x,y) = mean_tuple([grad(modelo, X_train[:,k], y_train[:,k]) for k in 1:size(X_train,2)])

# funciones de activacion
relu(x) = max.(0,x)
id(x) = x
softmax(x) = exp.(x) / sum(exp.(x))

softmax (generic function with 1 method)

In [75]:
mutable struct RedNeuronal{T<:Real}
	# Utilizaremos una red neuronal con tres capas con funciones de activacion ReLU,
	# identidad y softmax de 5 niveles.
	#
	# Nota: no es como Python, no hace falta definir una clase, ni tapoco
	# hace falta definir atributos como __init__ para poder definir el llamado a
	# el constructor de esta clase. Aca directamente llamamos
	# m = RedNeuronal(W1, b1, W2, b2)
	# y listo, con esto es suficiente para crear el modelo.
	#
	# El downside es que no podemos crear metodos bajo el struct RedNeuronal
	W1::Matrix{T} # l1 -> l2
	b1::Vector{T}
	W2::Matrix{T} # l2 -> l3
	b2::Vector{T}
end

# creamos el modelo con el constructor de Red Neuronal
Random.seed!(73)
W1 = randn(5, size(X_train, 2))
b1 = randn(5)
W2 = randn(size(y_train, 1), 5)
b2 = randn(size(y_train, 1 ))

model = RedNeuronal(W1, b1, W2, b2)

RedNeuronal{Float64}([1.2123689466501288 -1.8659948470385643 1.5359646820970976 -2.946693261135319; 0.07867339151355215 -0.5409383051722126 1.904244155888974 0.4000651169550351; … ; -1.4081384368973173 0.3566138060639536 0.758845608555424 -0.2462781339520417; 0.4644730607385192 -0.4580210630616379 -2.3363570247525765 0.6032439841236702], [0.036509117320801934, 2.0183941863348958, -1.4222121770628655, -0.02458224672669713, 0.5662594627526059], [0.9865857065343743 0.5569751877519062 … -0.013964807677120253 -1.1131349472530667; 0.5499123327973198 0.12834599681305806 … -0.7764680654747667 0.36181588423766176; -0.7411473285232308 -0.06050648910140387 … -0.7658544536287244 2.1530628891736723], [-0.14226841165752968, -1.0368122047759714, -0.14131008223303249])

In [77]:
X_train, y_train, X_test, y_test, classes = prepare_data(X, y)
W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2

([1.2123689466501288 -1.8659948470385643 1.5359646820970976 -2.946693261135319; 0.07867339151355215 -0.5409383051722126 1.904244155888974 0.4000651169550351; … ; -1.4081384368973173 0.3566138060639536 0.758845608555424 -0.2462781339520417; 0.4644730607385192 -0.4580210630616379 -2.3363570247525765 0.6032439841236702], [0.036509117320801934, 2.0183941863348958, -1.4222121770628655, -0.02458224672669713, 0.5662594627526059], [0.9865857065343743 0.5569751877519062 … -0.013964807677120253 -1.1131349472530667; 0.5499123327973198 0.12834599681305806 … -0.7764680654747667 0.36181588423766176; -0.7411473285232308 -0.06050648910140387 … -0.7658544536287244 2.1530628891736723], [-0.14226841165752968, -1.0368122047759714, -0.14131008223303249])

In [79]:
W1 * X_train .+ b1

5×120 Matrix{Float64}:
 -1.516      0.41072  -0.0415392  …  -1.75391   -1.84329   -0.253474
 -1.35911   -1.1061   -1.62163        4.5429     4.01707    4.13888
 -0.513142  -1.44864  -1.25785       -0.926415  -0.196279  -2.07961
  1.02424    1.59259   1.74668       -1.17754   -1.52106   -0.344671
  1.98462    2.29492   2.45229       -0.69366    0.214123  -0.997889

In [89]:
function forward_pass(model, X)
	# Implementar una funcion que para cada dato x retorne la prediccion segun
	# el modelo.
	W1, b1, W2, b2 = model.W1, model.b1, model.W2, model.b2
	z_1 = W1 * X .+ b1
	a_1 = relu(z_1)
	z_2 = W2 * X .+ b2
	a_2 = id(z_2) # redundante pero descriptivo
	y = softmax(a_2)
	return y
end

forward_pass (generic function with 2 methods)

In [90]:
forward_pass(model, X_train)

3×120 Matrix{Float64}:
 1.5328e-5   1.61439e-5  9.0161e-6   0.018923     …  0.00104544   0.00141996
 5.35901e-5  4.83369e-5  3.62203e-5  0.000386278     0.000104756  9.84745e-5
 0.00464099  0.00432043  0.00730502  1.83384e-5      0.000176276  0.00011035

In [ ]:
# TODO
function grad(model, x, y)
	# Implementar una funcion que calcule el gradiente de la funcion de perdida para
	# cada par de datos (x,y) utilizando backprop. Usarla para calcular el grad completo
	# usando la funcion grad_total de utils_tools.jl

end

In [ ]:
# TODO
function train!(model, train_set, step, max_iter)
	# La funcion debe retornar el modelo entrenado y un vector ocn los valores de la
	# funcion de perdida en cada iteracion
	# Opcional: Probar distintas funciones de activacion y comparar
	cross_entropy(y, y_hat) = -sum(y*log.(y_hat))
	prepare_data(train_set)
	for i in 1:max_iter
		# begin iter
		# correr forward pass -> y
		# calcular cross entropy -> loss
		# hacer backprop -> delta_C
		# actualizar? preguntar
		# end iter
	end
end

In [ ]:
# TODO
function results()
	# Implementar una funcion que retorne un grafico de los valores de la funcion
	# de perdida para cada iteracion y el rendimiento de nuestro modelo tanto en
	# los conjuntos de entrenamiento como en los de prueba, por ejemplo calculando
	#
	# 				 # favorables
	# 		       		---------------
	# 		      		# casos totales

end

In [ ]:
# Cada metodo debe retornar:
# 	Estado (convegio, maximas iteraciones alcanzadas, no convergio)
# 	Numero de iteraciones realizadas
# 	Valor final de ||grad(f)||
# TODO
function gradiente_const()
end


In [ ]:
# TODO
function gradiente_armijo()
end

In [ ]:
# TODO
function newton_raphson()
end

In [ ]:
# TODO
function gradiente_conj_no_lineal()
end

In [ ]:
# TODO
function bfgs()
end

In [ ]:
# TODO
function dolan_more()
	# construir perfil de desempeno de dolan & more para la metrica de iteraciones.
	# graficar y realizar un breve analisis.
	# sug: el primer paso consiste en ejecutar cada uno de los algoritmos sobre cada
	# problema y registrr el costo (iteraciones) de cada uno. Asi, podemos formar la
	# matriz C de costos. Luego se puede calcular una matriz R con la formula de r_sp
	# (ver clase de perfil de desempeno)
end